# P02b · Resume p02 from saved features (bench run only)

p02 finished extracting DINOv3 features (native and 0.3 m; outline and 3 m dilated pooling) and saved them to
`02_models/features/2026-11-20_dinov3_resolution/features.parquet`, then the Colab session ran out of memory in
the `bench run` step. This notebook runs **only** that step, in a fresh session, with the memory-lean
`bench run` (one setting in memory at a time).

* **Runtime:** CPU is enough (no GPU needed). Runtime → Change runtime type → **CPU**. Secret: `GH_TOKEN`.
* **Exploratory only:** development events are held out; confirmatory labels stay locked.
* **Time:** about 10–20 minutes (four reads of the features file, 20 probes, 2,000 bootstrap replicates).
* **Outputs:** this run's folder: `bench_run/` (scores, metrics, comparisons, `report.md`, manifest) and the
  executed notebook.

In [ ]:
NOTEBOOK_NAME = "p02b_resolution_benchmark_resume"

# Drive layout (FYP Google account). Keep in sync with src/damagesar_bench/drive.py
import os, datetime, re
from google.colab import drive
drive.mount("/content/drive")
PROJECT = "/content/drive/MyDrive/DamageSAR-Bench"
FOLDERS = ["00_admin", "01_data/raw/bright", "01_data/raw/instance_labels", "01_data/external",
           "01_data/derived", "02_models/checkpoints", "02_models/features", "03_notebooks/pipeline",
           "03_notebooks/checks", "03_notebooks/scratch", "04_outputs/runs", "04_outputs/reports"]
for f in FOLDERS:
    os.makedirs(f"{PROJECT}/{f}", exist_ok=True)
def run_dir(name):
    slug = re.sub(r"[^a-z0-9]+", "-", name.lower()).strip("-")
    p = f"{PROJECT}/04_outputs/runs/{datetime.date.today().isoformat()}_{slug}"
    os.makedirs(p, exist_ok=True)
    return p
RUN = run_dir(NOTEBOOK_NAME)
print("Drive layout ready under", PROJECT)
print("This run saves to:", RUN)

In [ ]:
# Install this repo's package (private repo: needs GH_TOKEN in Colab Secrets, read access to damagesar-bench)
from google.colab import userdata
GH = userdata.get("GH_TOKEN")
!git clone -q https://{GH}@github.com/ThinukaLehan/damagesar-bench.git /content/damagesar-bench 2>/dev/null || (cd /content/damagesar-bench && git pull -q)
!pip install -q -e /content/damagesar-bench
COMMIT = !git -C /content/damagesar-bench rev-parse --short HEAD
COMMIT = COMMIT[0]
# A running Colab kernel does not see an editable install until a restart: put src/ on the path.
import sys, importlib
if "/content/damagesar-bench/src" not in sys.path:
    sys.path.insert(0, "/content/damagesar-bench/src")
importlib.invalidate_caches()
import damagesar_bench
print("damagesar-bench at commit", COMMIT, "->", damagesar_bench.__file__)

In [ ]:
# The features p02 saved (do not re-extract)
import datetime, shutil, psutil
import pandas as pd
from damagesar_bench.config import load_events
from damagesar_bench.run import feature_settings
FEAT_DIR = f"{PROJECT}/02_models/features/2026-11-20_dinov3_resolution"
FEATS = f"{FEAT_DIR}/features.parquet"
assert os.path.exists(FEATS), f"missing {FEATS}: run p02 first"
events = load_events("/content/damagesar-bench/configs/events.yaml")
print(f"{FEATS}: {os.path.getsize(FEATS) / 1e9:.2f} GB")
print("settings:", feature_settings([FEATS]))
shutil.copy(f"{FEAT_DIR}/timing.csv", f"{RUN}/timing.csv")
display(pd.read_csv(f"{RUN}/timing.csv"))
print(f"RAM: {psutil.virtual_memory().total / 1e9:.1f} GB total, {psutil.virtual_memory().available / 1e9:.1f} GB free")

In [ ]:
# bench run: probes per held-out development event, per-event AUROC, event-level comparisons
from IPython.display import Markdown
from damagesar_bench.run import run_benchmark
from damagesar_bench.experiments.resolution import resolution_comparisons
RESOLUTIONS = [None, 0.3]
METHODS = ["outline", "dilated:3"]
comps = resolution_comparisons("dinov3_vitb16", RESOLUTIONS, METHODS)
out = run_benchmark([FEATS], comps, events, f"{RUN}/bench_run", n_boot=2000)
print(f"peak-safe: {psutil.virtual_memory().available / 1e9:.1f} GB free after the run")
display(Markdown(open(f"{out}/report.md", encoding="utf-8").read()))

In [ ]:
# Compact view for ADR 0001 round 3: AUROC per event and setting, and the spread of the gains
m = pd.read_csv(f"{out}/event_metrics.csv")
print(m.pivot_table(index=["encoder", "pooling"], columns="event", values="auroc").round(3).to_string())
c = pd.read_csv(f"{out}/comparisons.csv")
print(c[["label", "strength", "n_events", "estimate", "lower", "upper", "verdict"]].round(4).to_string(index=False))
import json, numpy as np
for _, r in c.iterrows():
    g = np.array(list(json.loads(r["gains"]).values()))
    print(f"{r['label']}: SD of per-event gains = {g.std(ddof=1):.4f}")

In [ ]:
# File the executed notebook (with outputs) into this run's folder and 03_notebooks/pipeline/
import json
from google.colab import _message
nb = _message.blocking_request("get_ipynb", timeout_sec=60)["ipynb"]
stamp = datetime.date.today().isoformat()
for target in (f"{RUN}/{NOTEBOOK_NAME}_executed.ipynb",
               f"{PROJECT}/03_notebooks/pipeline/{stamp}_{NOTEBOOK_NAME}.ipynb"):
    with open(target, "w") as fh:
        json.dump(nb, fh)
    print("saved", target)
print("run folder:", sorted(os.listdir(RUN)))